# A/B testing de rebates comerciales

**Objetivo:** diseñar y probar el pipeline estadístico de un experimento aleatorizado que mida si ofrecer un rebate después del rechazo inicial aumenta la conversión y el ingreso neto por cliente.

> Este notebook no estima el efecto causal real con el histórico existente. Los resultados observados del A/B son simulados y se identifican explícitamente como *dry run*. Para obtener evidencia empresarial deben reemplazarse por resultados capturados después de una asignación aleatoria real.

## 1. Diseño experimental

- **Población:** clientes contactables que rechazaron una oferta primaria.
- **Momento de aleatorización:** inmediatamente después del rechazo.
- **Control:** gestión habitual sin el rebate NBO experimental.
- **Tratamiento:** rebate recomendado por el motor NBO.
- **Estimando principal:** intención de tratar (ITT).
- **Métrica primaria:** ingreso neto incremental acumulado a 90 días por cliente asignado.
- **Métricas secundarias:** conversión, ARPU, churn, morosidad, reclamos y NPS.
- **Hipótesis:** $H_0: E[Y(1)-Y(0)]=0$ frente a $H_1: E[Y(1)-Y(0)\neq0$.

La aleatorización debe mantenerse estable por cliente. Si un asesor puede contaminar ambos tratamientos, se debe aleatorizar por asesor, tienda o turno y usar errores estándar agrupados.

In [1]:
from pathlib import Path
import hashlib
import math

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

def encontrar_raiz_repo(inicio=None):
    actual = Path(inicio or Path.cwd()).resolve()
    for candidato in [actual, *actual.parents]:
        if (candidato / 'AGENTS.md').exists() and (candidato / 'data').exists():
            return candidato
    raise FileNotFoundError('No se encontró la raíz del repositorio')

REPO_ROOT = encontrar_raiz_repo()
HISTORIAL_PATH = REPO_ROOT / 'data' / 'raw' / 'historial_campanias.csv'
CATALOGO_PATH = REPO_ROOT / 'data' / 'raw' / 'catalogo_ofertas_entrega.csv'
SEED = 20260826
print('Raíz:', REPO_ROOT)

Raíz: D:\hacka_movistar


## 2. Auditoría del histórico real

Los archivos de `data/raw/` se leen sin modificarlos. Antes de diseñar el experimento comprobamos si el histórico permite una comparación causal válida.

In [2]:
historial = pd.read_csv(HISTORIAL_PATH, low_memory=False)
catalogo = pd.read_csv(CATALOGO_PATH)
historial['fecha'] = pd.to_datetime(historial['fecha'], errors='coerce')

def a_bool(serie):
    return serie.astype(str).str.strip().str.lower().isin({'true', '1', 'si', 'sí'})

historial['es_rebate_bool'] = a_bool(historial['es_rebate'])
historial['elegible_mt_bool'] = a_bool(historial['elegible_mt'])
historial['resultado_norm'] = historial['resultado'].astype(str).str.strip().str.lower()

auditoria = pd.crosstab(
    historial['es_rebate_bool'].map({False: 'Oferta primaria', True: 'Rebate'}),
    historial['resultado_norm'],
    margins=True,
)
display(auditoria)
print(f'Registros: {len(historial):,}')
print(f'Clientes únicos: {historial.cliente_id.nunique():,}')
print(f'Periodo: {historial.fecha.min().date()} a {historial.fecha.max().date()}')

resultado_norm,aceptada,pendiente,rechazada,All
es_rebate_bool,,,,
Oferta primaria,95414,45494,111632,252540
Rebate,0,0,47572,47572
All,95414,45494,159204,300112


Registros: 300,112
Clientes únicos: 95,019
Periodo: 2026-01-10 a 2026-06-10


In [3]:
rebates_historicos = historial[historial['es_rebate_bool']]
positivos_rebate = rebates_historicos['resultado_norm'].eq('aceptada').sum()
print(f'Rebates históricos: {len(rebates_historicos):,}')
print(f'Rebates aceptados: {positivos_rebate:,}')
assert positivos_rebate == 0, 'La premisa documentada cambió: revisar el diseño antes de continuar.'
print('\nConclusión: el histórico no contiene variación de resultado dentro de rebates.')
print('Comparar rebate vs. no rebate produciría sesgo de selección y no identifica un efecto causal.')

Rebates históricos: 47,572
Rebates aceptados: 0

Conclusión: el histórico no contiene variación de resultado dentro de rebates.
Comparar rebate vs. no rebate produciría sesgo de selección y no identifica un efecto causal.


## 3. Cohorte elegible y asignación estable 50/50

Para el ensayo se toma el rechazo primario más reciente por cliente. La función hash reproduce siempre la misma asignación y evita depender del orden de las filas. En producción, la asignación debe persistirse antes de mostrar cualquier oferta.

In [4]:
cohorte = (
    historial.loc[
        (~historial['es_rebate_bool'])
        & historial['resultado_norm'].eq('rechazada')
        & historial['contactabilidad'].astype(str).str.lower().eq('contactado')
    ]
    .sort_values('fecha')
    .drop_duplicates('cliente_id', keep='last')
    .copy()
)

def asignacion_estable(cliente_id, semilla=SEED):
    texto = f'{semilla}|{cliente_id}'.encode('utf-8')
    entero = int.from_bytes(hashlib.sha256(texto).digest()[:8], 'big')
    return 'tratamiento_rebate_nbo' if entero % 2 else 'control_gestion_habitual'

cohorte['grupo_ab'] = cohorte['cliente_id'].map(asignacion_estable)
cohorte['tratamiento'] = cohorte['grupo_ab'].eq('tratamiento_rebate_nbo').astype(int)
cohorte['dry_run'] = True

display(cohorte['grupo_ab'].value_counts().rename('n').to_frame())
print(f'Cohorte experimental propuesta: {len(cohorte):,} clientes')

,n
grupo_ab,
control_gestion_habitual,33437
tratamiento_rebate_nbo,33258


Cohorte experimental propuesta: 66,695 clientes


## 4. Controles de calidad: SRM y balance basal

Un *Sample Ratio Mismatch* (SRM) detecta problemas de instrumentación si la distribución observada se aparta del 50/50. El balance no es una condición necesaria de la aleatorización, pero diferencias extremas pueden revelar errores.

In [5]:
n_trat = int(cohorte['tratamiento'].sum())
n_total = len(cohorte)
srm = stats.binomtest(n_trat, n_total, p=0.5, alternative='two-sided')
print(f'SRM p-value: {srm.pvalue:.4f}')
print('Resultado:', 'sin evidencia de SRM' if srm.pvalue >= 0.01 else 'ALERTA: revisar instrumentación')

def balance_categorico(df, columna):
    tabla = pd.crosstab(df[columna].fillna('NULO'), df['grupo_ab'], normalize='columns')
    tabla['diferencia_abs'] = (tabla.iloc[:, 0] - tabla.iloc[:, 1]).abs()
    return tabla.sort_values('diferencia_abs', ascending=False)

for columna in ['canal', 'tipo_cliente', 'elegible_mt_bool', 'tipo_oferta']:
    print(f'\nBalance: {columna}')
    display(balance_categorico(cohorte, columna).head(10))

x0 = cohorte.loc[cohorte.tratamiento.eq(0), 'antiguedad_meses'].dropna().astype(float)
x1 = cohorte.loc[cohorte.tratamiento.eq(1), 'antiguedad_meses'].dropna().astype(float)
sd_pool = math.sqrt((x0.var(ddof=1) + x1.var(ddof=1)) / 2)
smd_antiguedad = (x1.mean() - x0.mean()) / sd_pool
print(f'Diferencia estandarizada de antigüedad: {smd_antiguedad:.4f}')

SRM p-value: 0.4907
Resultado: sin evidencia de SRM

Balance: canal


grupo_ab,control_gestion_habitual,tratamiento_rebate_nbo,diferencia_abs
canal,,,
Call Out,0.1529,0.1475,0.0054
Call In,0.2000,0.2037,0.0037
Digital,0.3470,0.3493,0.0023
Tienda,0.3001,0.2994,0.0006



Balance: tipo_cliente


grupo_ab,control_gestion_habitual,tratamiento_rebate_nbo,diferencia_abs
tipo_cliente,,,
postpago,0.5407,0.5459,0.0052
NULO,0.0719,0.0683,0.0036
prepago,0.3874,0.3858,0.0016



Balance: elegible_mt_bool


grupo_ab,control_gestion_habitual,tratamiento_rebate_nbo,diferencia_abs
elegible_mt_bool,,,
True,0.1057,0.1058,0.0001
False,0.8943,0.8942,0.0001



Balance: tipo_oferta


grupo_ab,control_gestion_habitual,tratamiento_rebate_nbo,diferencia_abs
tipo_oferta,,,
paquete_adicional,0.1464,0.1499,0.0035
upgrade,0.1531,0.1514,0.0017
movistar_total,0.0553,0.0544,0.0009
equipo,0.1504,0.1498,0.0006
plan_movil,0.1953,0.1951,0.0003
plan_hogar,0.2994,0.2995,0.0001


Diferencia estandarizada de antigüedad: 0.0031


## 5. Potencia estadística y tamaño de muestra

El tamaño se calcula para una prueba bilateral de dos proporciones con asignación 50/50. Los parámetros deben reemplazarse por la conversión orgánica posterior al rechazo y el mínimo efecto relevante para negocio.

In [6]:
def muestra_por_brazo_proporciones(p_control, p_tratamiento, alpha=0.05, power=0.80):
    delta = abs(p_tratamiento - p_control)
    if delta == 0:
        return math.inf
    p_bar = (p_control + p_tratamiento) / 2
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    numerador = (
        z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
        + z_beta * math.sqrt(p_control * (1 - p_control) + p_tratamiento * (1 - p_tratamiento))
    ) ** 2
    return math.ceil(numerador / delta**2)

escenarios = []
for baseline in [0.05, 0.10, 0.15]:
    for uplift in [0.01, 0.02, 0.03]:
        n = muestra_por_brazo_proporciones(baseline, baseline + uplift)
        escenarios.append({
            'conversion_control': baseline,
            'uplift_absoluto': uplift,
            'n_por_brazo': n,
            'n_total': 2 * n,
        })
display(pd.DataFrame(escenarios))

,conversion_control,uplift_absoluto,n_por_brazo,n_total
0,0.0500,0.0100,8158,16316
1,0.0500,0.0200,2213,4426
2,0.0500,0.0300,1059,2118
3,0.1000,0.0100,14751,29502
4,0.1000,0.0200,3841,7682
5,0.1000,0.0300,1774,3548
6,0.1500,0.0100,20559,41118
7,0.1500,0.0200,5274,10548
8,0.1500,0.0300,2402,4804


## 6. Dry run: simulación del resultado futuro

Esta sección comprueba que la estimación y los controles funcionan. **No usa resultados observados de rebates.** Los supuestos son editables. El ingreso neto es un proxy porque el repositorio no contiene costos ni margen real.

In [7]:
PARAMETROS_SIMULACION = {
    'conversion_control': 0.10,
    'uplift_rebate': 0.02,
    'horizonte_meses': 3,
    'margen_contribucion': 0.45,
    'descuento_rebate': 0.10,
    'costo_contacto_control': 0.50,
    'costo_contacto_tratamiento': 2.00,
}

experimento = cohorte.merge(
    catalogo[['oferta_id', 'precio_mensual']], on='oferta_id', how='left', validate='many_to_one'
)
precio_mediano = catalogo['precio_mensual'].median()
experimento['precio_mensual'] = experimento['precio_mensual'].fillna(precio_mediano)

# Heterogeneidad basal plausible; el uplift causal configurado sigue siendo +2 pp.
ajuste_canal = experimento['canal'].map({
    'Call Out': 0.015, 'Call In': 0.010, 'Tienda': 0.005, 'Digital': -0.005
}).fillna(0.0)
p_control_individual = np.clip(
    PARAMETROS_SIMULACION['conversion_control']
    + ajuste_canal
    + 0.01 * experimento['elegible_mt_bool'].astype(int),
    0.01, 0.95,
)
experimento['prob_conversion_simulada'] = np.clip(
    p_control_individual + PARAMETROS_SIMULACION['uplift_rebate'] * experimento['tratamiento'],
    0.01, 0.99,
)
rng = np.random.default_rng(SEED + 1)
experimento['conversion_observada_simulada'] = (
    rng.random(len(experimento)) < experimento['prob_conversion_simulada']
).astype(int)

factor_descuento = np.where(
    experimento['tratamiento'].eq(1),
    1 - PARAMETROS_SIMULACION['descuento_rebate'],
    1.0,
)
costo_contacto = np.where(
    experimento['tratamiento'].eq(1),
    PARAMETROS_SIMULACION['costo_contacto_tratamiento'],
    PARAMETROS_SIMULACION['costo_contacto_control'],
)
experimento['ingreso_neto_90d_simulado'] = (
    experimento['conversion_observada_simulada']
    * experimento['precio_mensual']
    * factor_descuento
    * PARAMETROS_SIMULACION['horizonte_meses']
    * PARAMETROS_SIMULACION['margen_contribucion']
    - costo_contacto
)
display(pd.Series(PARAMETROS_SIMULACION, name='valor').to_frame())

,valor
conversion_control,0.1000
uplift_rebate,0.0200
horizonte_meses,3.0000
margen_contribucion,0.4500
descuento_rebate,0.1000
costo_contacto_control,0.5000
costo_contacto_tratamiento,2.0000


## 7. Estimación ITT e intervalos de confianza

Se informa el efecto absoluto, no solo el porcentaje relativo. La decisión principal se basa en ingreso neto por cliente asignado; conversión es secundaria.

In [8]:
def diferencia_proporciones(df, outcome='conversion_observada_simulada'):
    y0 = df.loc[df.tratamiento.eq(0), outcome].astype(float)
    y1 = df.loc[df.tratamiento.eq(1), outcome].astype(float)
    p0, p1 = y0.mean(), y1.mean()
    efecto = p1 - p0
    se = math.sqrt(p0 * (1 - p0) / len(y0) + p1 * (1 - p1) / len(y1))
    ci = (efecto - 1.96 * se, efecto + 1.96 * se)
    p_pool = (y0.sum() + y1.sum()) / (len(y0) + len(y1))
    se_pool = math.sqrt(p_pool * (1 - p_pool) * (1 / len(y0) + 1 / len(y1)))
    z = efecto / se_pool
    p_value = 2 * stats.norm.sf(abs(z))
    return {'control': p0, 'tratamiento': p1, 'efecto_abs': efecto, 'ci_95_inf': ci[0], 'ci_95_sup': ci[1], 'p_value': p_value}

def diferencia_medias(df, outcome):
    y0 = df.loc[df.tratamiento.eq(0), outcome].astype(float)
    y1 = df.loc[df.tratamiento.eq(1), outcome].astype(float)
    efecto = y1.mean() - y0.mean()
    se = math.sqrt(y0.var(ddof=1) / len(y0) + y1.var(ddof=1) / len(y1))
    dof_num = (y0.var(ddof=1) / len(y0) + y1.var(ddof=1) / len(y1)) ** 2
    dof_den = ((y0.var(ddof=1) / len(y0)) ** 2 / (len(y0) - 1)
               + (y1.var(ddof=1) / len(y1)) ** 2 / (len(y1) - 1))
    dof = dof_num / dof_den
    critico = stats.t.ppf(0.975, dof)
    t_stat = efecto / se
    p_value = 2 * stats.t.sf(abs(t_stat), dof)
    return {'control': y0.mean(), 'tratamiento': y1.mean(), 'efecto_abs': efecto, 'ci_95_inf': efecto-critico*se, 'ci_95_sup': efecto+critico*se, 'p_value': p_value}

resultados_itt = pd.DataFrame({
    'conversion': diferencia_proporciones(experimento),
    'ingreso_neto_90d_por_cliente': diferencia_medias(experimento, 'ingreso_neto_90d_simulado'),
}).T
resultados_itt['significativo_5pct'] = resultados_itt['p_value'] < 0.05
display(resultados_itt)

,control,tratamiento,efecto_abs,ci_95_inf,ci_95_sup,p_value,significativo_5pct
conversion,0.1075,0.1260,0.0185,0.0137,0.0234,0.0000,True
ingreso_neto_90d_por_cliente,10.2353,9.2815,-0.9539,-1.5263,-0.3815,0.0011,True


In [9]:
# Bootstrap no paramétrico del efecto en ingreso neto.
def bootstrap_diferencia_medias(df, outcome, repeticiones=2000, seed=SEED + 2):
    rng_local = np.random.default_rng(seed)
    y0 = df.loc[df.tratamiento.eq(0), outcome].to_numpy(float)
    y1 = df.loc[df.tratamiento.eq(1), outcome].to_numpy(float)
    efectos = np.empty(repeticiones)
    for i in range(repeticiones):
        efectos[i] = (
            rng_local.choice(y1, len(y1), replace=True).mean()
            - rng_local.choice(y0, len(y0), replace=True).mean()
        )
    return pd.Series({
        'efecto_promedio': efectos.mean(),
        'ci_95_inf': np.quantile(efectos, 0.025),
        'ci_95_sup': np.quantile(efectos, 0.975),
        'probabilidad_efecto_positivo': (efectos > 0).mean(),
    })

display(bootstrap_diferencia_medias(experimento, 'ingreso_neto_90d_simulado').to_frame('bootstrap'))

,bootstrap
efecto_promedio,-0.9619
ci_95_inf,-1.5304
ci_95_sup,-0.3922
probabilidad_efecto_positivo,0.0015


## 8. Heterogeneidad preespecificada

Los segmentos deben declararse antes de observar resultados. Estos cortes son exploratorios y no sustituyen el efecto ITT global; múltiples comparaciones aumentan los falsos positivos.

In [10]:
filas_segmento = []
for variable in ['canal', 'elegible_mt_bool']:
    for valor, subgrupo in experimento.groupby(variable, dropna=False):
        if subgrupo['tratamiento'].nunique() < 2 or len(subgrupo) < 100:
            continue
        estimacion = diferencia_proporciones(subgrupo)
        filas_segmento.append({
            'variable': variable, 'segmento': valor, 'n': len(subgrupo), **estimacion
        })
segmentos = pd.DataFrame(filas_segmento)
display(segmentos.sort_values(['variable', 'segmento']))

,variable,segmento,n,control,tratamiento,efecto_abs,ci_95_inf,ci_95_sup,p_value
0,canal,Call In,13462,0.1110,0.1358,0.0248,0.0137,0.0359,0.0000
1,canal,Call Out,10020,0.1193,0.1284,0.0091,-0.0038,0.0220,0.1652
2,canal,Digital,23220,0.0991,0.1170,0.0179,0.0099,0.0258,0.0000
3,canal,Tienda,19993,0.1088,0.1287,0.0199,0.0109,0.0289,0.0000
4,elegible_mt_bool,False,59643,0.1069,0.1255,0.0185,0.0134,0.0237,0.0000
5,elegible_mt_bool,True,7052,0.1121,0.1305,0.0184,0.0032,0.0337,0.0178


## 9. Contrato mínimo de datos para producción

Cada asignación debe persistir una sola vez con estas columnas:

| Campo | Propósito |
|---|---|
| `experiment_id`, `assignment_id` | Trazabilidad e idempotencia |
| `cliente_id`, `timestamp_asignacion` | Unidad y momento de asignación |
| `grupo_ab`, `unidad_cluster` | Tratamiento y posible agrupación |
| `oferta_inicial_id`, `rebate_id` | Exposición comercial |
| `rebate_mostrado` | Cumplimiento; no debe redefinir el ITT |
| `conversion_30d`, `conversion_90d` | Resultados comerciales |
| `ingreso_neto_90d`, `margen_90d` | Resultado económico |
| `churn_90d`, `mora_90d`, `reclamo_90d`, `nps` | Guardrails |

No se debe asignar tratamiento a partir de variables posteriores al rechazo ni eliminar del análisis a quienes no recibieron finalmente el speech.

## 10. Regla de decisión

Se recomienda desplegar el rebate NBO si, al cierre predefinido del experimento:

1. El intervalo de confianza del ingreso neto incremental a 90 días excluye cero por el lado positivo.
2. El efecto supera el mínimo económicamente relevante definido antes del ensayo.
3. No empeoran materialmente churn, mora, reclamos o NPS.
4. No existe SRM ni evidencia de contaminación o instrumentación defectuosa.

Una mejora significativa en conversión no basta si el descuento reduce el margen neto. Los resultados de este notebook son una validación técnica simulada; la conclusión causal solo puede emitirse con resultados reales del A/B.